# Homework 2

In [1]:
# question 1
from embedder import Embedder

embed=Embedder()

In [2]:
query='How does approximate nearest neighbor search work?'
query_embedded=embed.encode(query)

In [3]:
query_embedded

array([-2.05820344e-02, -1.40458849e-02,  3.02994061e-02, -5.40378445e-02,
        7.18781100e-02, -2.79537512e-02, -5.03093823e-02, -1.27217287e-02,
        4.08207902e-02, -2.60037446e-02,  3.05458646e-02,  4.21485309e-02,
        8.09861910e-02, -6.93957355e-02, -1.30190518e-01, -6.39247860e-02,
        4.81059741e-02,  1.60095554e-02, -5.22432468e-02, -7.13635281e-02,
       -3.83859209e-03,  2.48125508e-02,  4.40211692e-02, -3.45579077e-02,
        1.52686257e-02,  7.61533350e-03,  5.38177679e-02,  1.18252557e-02,
        1.32434005e-02,  2.89461963e-02,  6.64912054e-03,  7.04788357e-02,
        6.19290508e-02,  2.11051780e-02, -7.33482889e-02,  2.84129213e-02,
       -3.72108635e-02,  6.22799314e-02, -4.86973136e-02,  4.49663910e-02,
       -2.59481060e-02,  2.04324269e-02,  1.79650458e-02,  1.02705602e-02,
        2.74898095e-03,  2.84324992e-02, -3.30318167e-02,  6.70969402e-02,
       -1.56520555e-02, -8.51907638e-02, -1.24307135e-01,  4.32504103e-02,
       -5.64433014e-02,  

In [4]:
query_embedded[0]

np.float64(-0.02058203437252893)

In [5]:
# load the data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
documents

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [6]:
documents[1]

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install uv` if you p

In [9]:
len(documents)

72

In [16]:
# question 2
# cosine similarity

documents[22]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [17]:
content=documents[22]['content']
content

'# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done so far is ex

In [18]:
embedded_content=embed.encode(content)


In [19]:
# compute cosine similarity
embedded_content.dot(query_embedded)

np.float64(0.36107026789538205)

In [21]:
# question 3
# chunking and search by hand

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

In [23]:
chunks[0]['content']

'# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language 

In [27]:
len(chunks)

295

In [24]:
import numpy as np

In [25]:
np.stack

TypeError: dispatcher for __array_function__ did not return an iterable

In [ ]:
tokenizer_path='models/Xenova/all-MiniLM-L6-v2/tokenizer.json'

In [30]:

for chunk in chunks:
    embedded_chunk_content=embed.encode(chunk['content'])
    X=np.stack(embedded_chunk_content)

In [32]:
len(X),X.shape

(384, (384,))

In [ ]:
batch_size=60
for i in range(0,len(chunks),batch_size:
    chunks_batch=chunks[i:i+batch_size]
    embedded_chunks_batch=embed.encode_batch(chunks_batch)

In [36]:
vector_chunk=[]
for chunk in chunks:
    embedded_chunk=embed.encode(chunk['content'])
    vector_chunk.extend([embedded_chunk])

In [37]:
vector_chunk,len(vector_chunk)

([array([-8.75646898e-02,  1.83638019e-02, -8.12242020e-02,  7.68664297e-03,
         -1.29271640e-02,  1.16904850e-02,  5.30512782e-03, -3.12753599e-02,
         -8.49467116e-02, -7.08059461e-02,  2.45027996e-02,  2.42203869e-02,
          8.10846478e-02, -1.54682351e-02, -4.05255447e-02,  6.04803055e-02,
          4.46619172e-02,  7.89197036e-02,  1.57477445e-02, -8.72309219e-02,
         -1.97475348e-02,  6.54039725e-02,  5.31242067e-02, -4.77922113e-02,
         -3.76264882e-02,  1.85600032e-02, -1.93437972e-02, -2.97848491e-02,
          9.46586812e-02,  4.79098808e-03,  6.11718165e-02,  1.12155455e-01,
         -6.47929541e-03,  6.07943713e-02, -1.40041985e-02,  5.97074838e-02,
         -5.57415284e-02,  1.83090183e-02, -4.91055788e-02, -6.45832049e-02,
         -2.91399273e-02,  2.86424004e-02, -1.50477997e-02, -2.62999281e-02,
          5.41334897e-02, -2.22229951e-02, -4.69662066e-02, -9.27702628e-03,
          5.74242027e-02, -4.87887416e-02, -9.17564003e-02, -3.97100080e-02,

In [40]:
vector_chunk[0].shape,query_embedded.shape

((384,), (384,))

In [41]:
X=np.array(vector_chunk)
X.shape

(295, 384)

In [42]:
scores = X.dot(query_embedded)

In [44]:
top5=np.argsort(-scores)[:5]

In [45]:
top5

array([ 94,  14, 162,  85, 253])

In [46]:
scores[top5]

array([0.64890164, 0.55103467, 0.40656068, 0.40618203, 0.40610592])

In [47]:
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [48]:
# question 4
from minsearch import VectorSearch
vs_index=VectorSearch(
)


In [49]:
vs_index.fit(X,chunks)

In [50]:
query='What metric do we use to evaluate a search engine?'

In [51]:
embedded_query=embed.encode(query)

In [52]:
vs_index.search(
    embedded_query,
    num_results=3
)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [53]:
# question 5

# VectorSearch
vs_index_comp=VectorSearch()
vs_index_comp.fit(X,chunks)

In [54]:
query='How do I store vectors in PostgreSQL?'

In [55]:
embedded_query=embed.encode(query)

In [56]:
# search 
vector_search_results=vs_index_comp.search(
    embedded_query,
    num_results=5
)

In [57]:
vector_search_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [59]:
# text search 
from minsearch import Index

text_index=Index(
    text_fields=['content']
)
text_index.fit(chunks)

In [61]:
# search 
text_search_results=text_index.search(
    query,
    num_results=5
)

In [62]:
text_search_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [63]:
# question 6 
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [64]:
q="How do I give the model access to tools?"

In [65]:
embedded_q=embed.encode(q)

In [66]:
vs_results=vs_index_comp.search(
    embedded_q,
    num_results=5
)

In [67]:
vs_results

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

In [68]:
ts_results=text_index.search(
    q,
    num_results=5
)

In [69]:
ts_results

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can

In [70]:
results = rrf([vs_results, ts_results])

In [71]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

In [72]:
results[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca